In [ ]:
class Logistic_reg:
    """
    Binary logistic regression fit via batch gradient ascent.

    Parameters
    --------------------------------------------
    X: ndarray of shape (n_samples, n_features)
        Training feature matrix (should be standardized beforehand).
    y: ndarray of shape (n_samples,)
        Binary target labels (0-1)

    Attributes
    --------------------------------------------
    n_samples: int
        number of training samples.
    n_features: int
        number of features (excluding intercept).
    theta_new: ndarray of shape (n_features + 1)
        Fitted coefficients (index 0 is the intercept), set after calling gradient_descent
    x_ext: ndarray of shape (n_samples, n_features + 1)
        Training features with an added intercept column of ones
    """

    def __init__(self,X, Y):
        self._x = X
        self._y = Y
        self.n_samples, self.n_features = self._x.shape

    def sigmoid(self, z):
        """
        Compute the logistic signoid function element-wise.

        Parameters
        ------------------------------------------
        z: ndarray
            Linear predictor values.

        Returns
        ------------------------------------------
        ndarray values in 0-1, same shape as z
        """
        return np.where( z>=0, 
                            1 / (1 + np.exp(-z)),
                            np.exp(z) / (1 + np.exp(z)))

    def derivate_log_ver(self, theta):
        """
        Compute the gradient of the log-likelihood w.r.t theta.

        Parameters
        ------------------------------------------
        Theta: ndarray of shape (n_features + 1)
            current coefficient estimates (intercept first)

        Returns
        ------------------------------------------
        ndarray of shape (n_features + 1)
        gradient vector, averaged over samples

        Notes
        ------------------------------------------
        Also sets self.x_ext, the feature matrix with intercept column, as side effect
        """
        self.x_ext = np.hstack((np.ones((self.n_samples, 1)),self._x))
        p = self.sigmoid(self.x_ext @ theta)
        return (self.x_ext.T @ (self._y - p)) / self.n_samples

    def gradient_descent(self, max_iter= 1000, learning_rate = 0.001, tol=0.01):
        """
        Fit theta via batch gradient ascent on the log-likelihood

        Parameters
        ------------------------------------------
        Max_iter: int, defaul=1000
            maximum number of iterations.
        learning_rate: float, default=0.001
            step sieze for each update
        tol: float, default= 0.01
            convergance threshold on the max coordinate-wise change in theta between iterations

        returns
        ------------------------------------------
        theta_new: ndarray of shape (n_feaure + 1)
            fitted coefficients
        success_flag: int
            1 if the convergence was reached before max_iter, else 0
        history: ndarray
            per-iteration max coordinate change, up to convergence
        """
        beta = np.random.randn(self.n_features)
        theta_initial = np.hstack(([0.0], beta))

        theta_old = theta_initial.copy()
        self.theta_new = theta_old
        success_flag = 0
        history = np.full(max_iter,np.nan)


        for i in range(max_iter):
            grad = self.derivate_log_ver(theta_old)

            theta_new = theta_old + learning_rate * grad

            diff = np.max(abs(theta_new - theta_old))
            history[i] = diff

            if diff < tol:
                success_flag = 1
                break
            theta_old = theta_new

        self.theta_new = theta_old
        return self.theta_new, success_flag, history[~np.isnan(history)]

    def prob_predict(self, X):
        """Predict class-1 probabilities for new samples.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix (must use the same scaling as training data).

        Returns
        -------
        ndarray of shape (n_samples,)
            Predicted probabilities in (0, 1).
        """
        x_ext = np.hstack((np.ones((X.shape[0], 1)),X))
        z = x_ext @ self.theta_new
        return self.sigmoid(z)
    
    def predict(self, X ,threshold = 0.5):
        """Predict binary class labels for new samples.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix (must use the same scaling as training data).
        threshold : float, default=0.5
            Decision threshold applied to predicted probabilities.

        Returns
        -------
        ndarray of shape (n_samples,)
            Predicted labels (0 or 1).
        """
        
        probs = self.prob_predict(X)

        return (probs > threshold).astype(int)

In [ ]:
# Practical case

import numpy as np
import pandas as pd

# Importing cleaned data (titanic case)
df = pd.read_csv('/Users/samuelsantiagosuarezpaez/Downloads/train_and_test2.csv')

# Minimal cleaning data
df.drop(columns=['zero.1',
       'zero.2', 'zero.3', 'zero.4', 'zero.5', 'zero.6', 'zero.7',
       'zero.8', 'zero.9', 'zero.10', 'zero.11', 'zero.12', 'zero.13',
       'zero.14', 'zero.15', 'zero.16', 'zero.17',
       'zero.18'],inplace=True)

df.dropna(inplace=True)

# Library for splitting data into training and testing sets
from sklearn.model_selection import train_test_split

X, y = df.drop(columns=['2urvived']), df['2urvived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


# Standardization of data

X_std = X_train.std(axis=0)
X_mean = X_train.mean(axis=0)
X_train_scaled = (X_train - X_mean) / X_std

# Drop constant columns before scaling
constant_cols = X_std[X_std == 0].index
X_train = X_train.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)
X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std


0


In [ ]:
import warnings
warnings.filterwarnings("error")  

# Prediction

try:
    modelo_logistic = Logistic_reg(X_train_scaled, y_train)
    theta_final, success, history = modelo_logistic.gradient_descent(
        max_iter=100000, learning_rate=0.1, tol=1e-6
    )
except Warning as w:
    print("Caught:", w)

predictions = modelo_logistic.predict(X_test_scaled,threshold=0.3)
print(predictions)

[0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 1 0 0 1 0 0 1
 0 0 1 1 0 1 0 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 0 1 1 1 1 1 1 1 0 1 1 0 0
 1 0 1 1 1 0 1 0 0 0 0 0 0 0 1 1 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0
 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1
 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 1 0 0 1
 0 0 0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 1 0 0 0 1 1
 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1 1
 0 1 0 1 1 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 1 1 0 0 1 0 0
 0 1 0 1 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 1
 0 1 1 1 0 0 1 0 1 1 1 0 0 0 0 1 0 0 0 1 1 0 0]


In [214]:
print(theta_final)
print(np.isnan(theta_final).any(), np.isinf(theta_final).any())

probs = modelo_logistic.prob_predict(X_test_scaled)
print(probs.min(), probs.max(), probs.mean())
print(success)
print(history[:10])
print(history[-10:])

[-1.75512844 -1.23366053 -0.42024931 -0.02067921  1.09857777 -0.36327476
  0.04308348 -0.8670223  -0.08161899]
False False
0.0005562319898078062 0.9639883711961926 0.2529366413130849
1
[0.02301402 0.02262934 0.0222503  0.02187684 0.02150894 0.02114655
 0.02078961 0.02043809 0.02009192 0.01975106]
[1.03049544e-06 1.02690221e-06 1.02332154e-06 1.01975336e-06
 1.01619765e-06 1.01265436e-06 1.00912344e-06 1.00560485e-06
 1.00209854e-06 9.98604483e-07]


In [215]:
print(success)                 
print(len(history))            
print(history[-10:])           
print(theta_final)

1
2226
[1.03049544e-06 1.02690221e-06 1.02332154e-06 1.01975336e-06
 1.01619765e-06 1.01265436e-06 1.00912344e-06 1.00560485e-06
 1.00209854e-06 9.98604483e-07]
[-1.75512844 -1.23366053 -0.42024931 -0.02067921  1.09857777 -0.36327476
  0.04308348 -0.8670223  -0.08161899]


In [218]:
test_accuracy = np.mean(predictions == y_test)
print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")


Test Accuracy: 82.44%


In [217]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

print(confusion_matrix(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))
print("F1:", f1_score(y_test, predictions))

[[239  38]
 [ 31  85]]
Precision: 0.6910569105691057
Recall: 0.7327586206896551
F1: 0.7112970711297071
